In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Sirifort_Delhi_CPCB_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,229.0,176.0,172.0,95.0,88.0,87.0,44.0,82.0,136.0,136.0,360.0,387.0
1,2,345.0,171.0,231.0,88.0,66.0,106.0,62.0,85.0,140.0,140.0,369.0,376.0
2,3,390.0,185.0,173.0,157.0,94.0,112.0,110.0,74.0,133.0,141.0,498.0,320.0
3,4,361.0,216.0,122.0,108.0,82.0,253.0,147.0,85.0,132.0,160.0,411.0,329.0
4,5,350.0,222.0,121.0,164.0,172.0,251.0,99.0,NaN,110.0,171.0,470.0,310.0
5,6,425.0,240.0,128.0,164.0,246.0,241.0,59.0,NaN,113.0,191.0,414.0,292.0
6,7,395.0,275.0,153.0,165.0,152.0,260.0,63.0,NaN,97.0,181.0,392.0,300.0
7,8,398.0,128.0,216.0,191.0,129.0,172.0,47.0,107.0,87.0,143.0,406.0,295.0
8,9,471.0,230.0,106.0,205.0,191.0,144.0,52.0,115.0,39.0,NaN,437.0,303.0
9,10,432.0,194.0,172.0,171.0,194.0,123.0,58.0,135.0,27.0,155.0,284.0,319.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   29 non-null     float64
 3   March      32 non-null     float64
 4   April      34 non-null     float64
 5   May        34 non-null     float64
 6   June       34 non-null     float64
 7   July       34 non-null     float64
 8   August     29 non-null     float64
 9   September  31 non-null     float64
 10  October    35 non-null     float64
 11  November   33 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Convert all columns except 'Day' to numeric values
for col in df.columns:
    if col != 'Day':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values with column mean
df_filled = df.fillna(df.mean(numeric_only=True))

In [8]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [9]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,229.0,176.0,172.0,95.0,88.0,87.000000,44.000000,82.000000,136.0,136.0,360.0,387.0
1,2,345.0,171.0,231.0,88.0,66.0,106.000000,62.000000,85.000000,140.0,140.0,369.0,376.0
2,3,390.0,185.0,173.0,157.0,94.0,112.000000,64.176471,74.000000,133.0,141.0,498.0,320.0
3,4,361.0,216.0,122.0,108.0,82.0,117.970588,64.176471,85.000000,132.0,160.0,411.0,329.0
4,5,350.0,222.0,121.0,164.0,172.0,117.970588,99.000000,101.655172,110.0,171.0,470.0,310.0
